# 01 Build the prompt set

Samples 3 harmful behaviors per category from JailbreakBench (JBB-Behaviors), 30 in total, plus the matched harmless version of each. Attaches the published PAIR jailbreak prompt for each behavior where one exists.

Runs on CPU with Internet on. Needs a Hugging Face token as `HF_TOKEN` (a Kaggle secret, or an environment variable elsewhere).

Output: `jbb_test_english.csv`

In [ ]:
import os
import json, requests
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except ImportError:
    login(token=os.environ.get("HF_TOKEN"))

N_PER_CATEGORY = 3
RANDOM_STATE = 42
OUT_CSV = "jbb_test_english.csv"

SEVERITY = {
    "Disinformation": "low", "Harassment/Discrimination": "low",
    "Expert advice": "low", "Economic harm": "low",
    "Privacy": "medium", "Fraud/Deception": "medium",
    "Government decision-making": "medium", "Sexual/Adult content": "medium",
    "Malware/Hacking": "high", "Physical harm": "high",
}

ds = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors")
harmful = ds["harmful"].to_pandas().assign(type="harmful")
benign = ds["benign"].to_pandas().assign(type="benign")
print("JBB harmful:", len(harmful), "benign:", len(benign))

sampled = (harmful.groupby("Category", group_keys=False)
           .apply(lambda g: g.sample(min(N_PER_CATEGORY, len(g)), random_state=RANDOM_STATE)))
sample_behaviors = set(sampled["Behavior"])

df = pd.concat([harmful, benign], ignore_index=True)
df = df[df["Behavior"].isin(sample_behaviors)]
df = df.rename(columns={"Goal": "english", "Behavior": "behavior", "Category": "category"})
df["severity"] = df["category"].map(SEVERITY).fillna("unknown")
df["pair_id"] = df["behavior"].factorize()[0]
df = df[["pair_id", "category", "severity", "behavior", "type", "english"]]

tree = requests.get("https://api.github.com/repos/JailbreakBench/artifacts/git/trees/main?recursive=1").json()
paths = [t["path"] for t in tree.get("tree", [])
         if "PAIR" in t["path"] and "vicuna" in t["path"].lower() and t["path"].endswith(".json")]
print("PAIR artifact file:", paths)

pair_df = pd.DataFrame(columns=["behavior", "goal", "pair_prompt"])
if paths:
    raw = f"https://raw.githubusercontent.com/JailbreakBench/artifacts/main/{paths[0]}"
    artifact = requests.get(raw).json()
    pair_df = pd.DataFrame([{"behavior": j.get("behavior"), "goal": j.get("goal"), "pair_prompt": j.get("prompt")}
                            for j in artifact["jailbreaks"]])
    pair_df = pair_df[pair_df["pair_prompt"].notna()]
print("PAIR prompts available:", len(pair_df))

df = df.merge(pair_df[["behavior", "pair_prompt"]], on="behavior", how="left")
if df["pair_prompt"].notna().sum() == 0 and len(pair_df):
    df = df.drop(columns="pair_prompt").merge(
        pair_df[["goal", "pair_prompt"]].rename(columns={"goal": "english"}), on="english", how="left")

df["pair_prompt"] = df["pair_prompt"].fillna("")
df.loc[df["type"] == "benign", "pair_prompt"] = ""
df = df.sort_values(["pair_id", "type"]).reset_index(drop=True)

harm = df[df["type"] == "harmful"]
print("\nSampled behaviors:", len(sample_behaviors), "| total rows:", len(df))
print("Pairs missing a benign twin:", df.groupby("pair_id")["type"].nunique().lt(2).sum())
print("Harmful prompts with a PAIR jailbreak:", (harm["pair_prompt"] != "").sum(), "of", len(harm))
print(df.groupby(["category", "type"]).size().unstack(fill_value=0))

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("\nSaved", OUT_CSV)

## Translation step (done outside the notebook)

Google Translate blocks requests from Kaggle's IP, so translation was done in Google Sheets:

1. Import `jbb_test_english.csv` into a sheet. Columns A to G are `pair_id, category, severity, behavior, type, english, pair_prompt`.
2. Add three columns with these exact headers:
   - H `telugu`: `=GOOGLETRANSLATE(F2, "en", "te")`
   - I `pair_telugu`: `=IF(G2="", "", GOOGLETRANSLATE(G2, "en", "te"))`
   - J `back_translation`: `=GOOGLETRANSLATE(H2, "te", "en")` (meaning check)
3. Fill the formulas down, download as CSV, and set `INPUT_CSV` in notebook 02 to its path.

Google Translate output changes over time, so a rerun will not give identical Telugu. The translated CSV used in this study is published with the dataset.